##### Libraries

In [1]:
import os, sys
env = r"C:\Users\Costanza\anaconda3\envs\Dati_climatici"
os.environ["ECCODES_DIR"] = env + r"\Library"
os.add_dll_directory(env + r"\Library\bin")
os.environ["PATH"] = env + r"\Library\bin;" + os.environ["PATH"]
import eccodes
import earthkit.data

print(sys.executable)
print(eccodes.codes_get_api_version())

c:\Users\Costanza\anaconda3\envs\Dati_climatici\python.exe
2.50.0


In [2]:
import json
import os
import earthkit.data
import cdsapi
import xarray as xr
import pandas as pd
import numpy as np
import cftime
import pprint
import cdsapi
from IPython.display import display
import sys
import pickle
import matplotlib.pyplot as plt
from scipy.stats import genextreme as gev
import reverse_geocoder as rg
import folium
import math
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
pio.renderers.default = "notebook"

import geopandas as gpd
import plotly.express as px

from pathlib import Path
from skgstat import Variogram

from shapely import contains_xy



##### Functions

In [3]:
def plot_semivariograms(semivar_mu, semivar_ls):

    def get_variogram_data(semivar):
        bins = semivar.bins
        experimental = semivar.experimental

        x_fit = np.linspace(0, bins[-1], 300)
        y_fit = semivar.fitted_model(x_fit)

        rng, sill, nugget = semivar.parameters

        return bins, experimental, x_fit, y_fit, rng, sill, nugget

    (bins_mu, exp_mu, xfit_mu, yfit_mu,
     rng_mu, sill_mu, nugget_mu) = get_variogram_data(semivar_mu)

    (bins_ls, exp_ls, xfit_ls, yfit_ls,
     rng_ls, sill_ls, nugget_ls) = get_variogram_data(semivar_ls)

    fig = make_subplots(
        rows=1,
        cols=2,
        subplot_titles=("Semivariogram — Mean", "Semivariogram — log(Standard deviation)"),
        horizontal_spacing=0.10
    )

    color_mu = "#1A69AA"
    color_ls = "#108110"

    for col, (bins, exp, xfit, yfit,
              rng, sill, nugget,
              color, label) in enumerate([

        (bins_mu, exp_mu, xfit_mu, yfit_mu,
         rng_mu, sill_mu, nugget_mu,
         color_mu, "Mean"),

        (bins_ls, exp_ls, xfit_ls, yfit_ls,
         rng_ls, sill_ls, nugget_ls,
         color_ls, "log(std)")

    ], start=1):

        # Punti sperimentali
        fig.add_trace(
            go.Scatter(
                x=bins,
                y=exp,
                mode="markers",
                marker=dict(
                    size=7,
                    color=color,
                    opacity=0.85,
                    line=dict(color="white", width=0.8),
                ),
                name=f"Empirical {label}",
                legendgroup=label,
            ),
            row=1,
            col=col,
        )

        # Curva teorica
        fig.add_trace(
            go.Scatter(
                x=xfit,
                y=yfit,
                mode="lines",
                line=dict(color=color, width=2.5),
                name=f"Exponential fit {label}",
                legendgroup=label,
            ),
            row=1,
            col=col,
        )

        # Sill
        fig.add_hline(
            y=sill,
            row=1,
            col=col,
            line_dash="dash",
            line_color="gray",
        )

        # Parametri
        fig.add_annotation(
            x=0.98,
            y=0.02,
            xref="x domain" if col == 1 else "x2 domain",
            yref="y domain" if col == 1 else "y2 domain",
            text=(
                f"<b>Range</b>: {rng/1000:.1f} km<br>"
                f"<b>Sill</b>: {sill:.3f}<br>"
                f"<b>Nugget</b>: {nugget:.3f}"
            ),
            showarrow=False,
            align="right",
            bgcolor="rgba(255,255,255,0.85)",
            bordercolor=color,
            borderwidth=1,
            font=dict(size=11),
        )

    fig.update_layout(
        width=950,
        height=430,
        template="plotly_white",
        title=dict(
            text="Spatial Semivariograms",
            x=0.5,
            font=dict(size=18),
        ),
        legend=dict(
            orientation="h",
            x=0.5,
            xanchor="center",
            y=-0.18,
        ),
    )

    fig.update_xaxes(
        title="Distance (km)",
        tickformat=".0f",
    )

    # Mostra le distanze in km
    fig.update_xaxes(
        tickvals=np.linspace(0, max(bins_mu.max(), bins_ls.max()), 6),
        ticktext=[
            f"{v/1000:.0f}"
            for v in np.linspace(0, max(bins_mu.max(), bins_ls.max()), 6)
        ],
    )

    fig.update_yaxes(title="Semivariance")

    fig.show()

def download_cds_dataset(dataset, request):
    """
    Download a CDS dataset and return it as an xarray Dataset.
    This wrapper works for both CORDEX-style and ERA5-Land-style requests.
    """
    ds = earthkit.data.from_source("cds", dataset, request)
    return ds.to_xarray(xarray_open_mfdataset_kwargs=dict(decode_cf=True, decode_times=True))

def download_dfs(requests, dictionay_name, dataset="projections-cordex-domains-single-levels"):
    """
    Function to "download" as a temporary file with Earthkit and add the orography/variables of several climate models to the dictionary (dfs_orography or dfs_models).
    :param requests: the parameters of the request (those indicated in the Copernicus dataset: names RCM, GCM, resolution, etc.).
    Example of a request
    irnet_orography = {
    "domain": "europe",
    "experiment": "historical",
    "horizontal_resolution": "0_11_degree_x_0_11_degree",
    "temporal_resolution": "fixed",
    "variable": ["orography"],
    "gcm_model": "ichec_ec_earth",
    "rcm_model": "knmi_racmo22e",
    "ensemble_member": "r1i1p1"
    }
    :param dictionary_name: the name of the dictionary to save the dataset in.
    :param dataset: CDS dataset identifier.
    """
    for key_name, request in requests:
        try:
            ds = download_cds_dataset(dataset, request)
            dictionay_name[key_name] = ds
            print(f"Dataset '{key_name}' downloaded and added to dictionary")

            # Convert tasmax from Kelvin to Celsius
            if "tasmax" in ds:
                ds["tasmax"] = ds["tasmax"] - 273.15
                ds["tasmax"].attrs["units"] = "°C"
            # Convert tasmin from Kelvin to Celsius
            if "tasmin" in ds:
                ds["tasmin"] = ds["tasmin"] - 273.15
                ds["tasmin"].attrs["units"] = "°C"
            # Convert pr from kg*m-2*s-1 to mm
            if "pr" in ds:
                ds["pr"] = ds["pr"] * 100000
                ds["pr"].attrs["units"] = "mm"
            # Convertion not needed for wind speed (already in m*s-1)

        except Exception as e:
            print(f"Error with the dataset '{key_name}': {e}")

def run_all_models(file_json, variable):
    """
    Load climate model configurations from a JSON file, download the corresponding
    datasets for each model and time interval, and return all results in a structured
    dictionary.

    This function reads a JSON file containing multiple climate model specifications
    (GCM, RCM, ensemble member, and temporal ranges). For each model, it iterates
    through the defined start/end year pairs, builds the request parameters, and
    downloads the associated datasets using `download_dfs()`. All downloaded data
    are organized into a nested dictionary structure.

    Parameters:
        file_json (str):
            Path or filename of the JSON file containing model configurations (e.g., "tasmax.json").
        variable (str):
            Name of the climate variable to download (e.g., "maximum_2m_temperature_in_the_last_24_hours"
                                                            "minimum_2m_temperature_in_the_last_24_hours"
                                                            "mean_precipitation_flux"
                                                            "10m_wind_speed").
    Returns:
        dict:
            A dictionary where each key corresponds to a model name, and each value
            contains another dictionary mapping each temporal chunk (start_year–end_year)
            to the downloaded dataset.
    """

    dfs_models = {}                                                  # Main dictionary
    base_dir = os.getcwd()                                           # base_dir = os.getcwd() obtain current folder
    with open(os.path.join(base_dir, file_json)) as f:
        models = json.load(f)                                        # Read models from json file
    for model in models:                                             # Iterate through the models and download the data
        gcm = models[model]["gcm"]
        rcm = models[model]["rcm"]
        # ensemble_orography = models[model]["ensemble_orography"]
        ensemble_model = models[model]["ensemble_model"]
        start_years = models[model]["start_years"]
        end_years = models[model]["end_years"]
        dfs_chunks = {}                                              # Dictionary to store temporal chunks. Chunks can be: 1950-1950,1951-1955,...,1971-1975

        for sy, ey in zip(start_years, end_years):                   # Loop over each temporal chunk

            request_models = [
                (
                    f"{model}_{sy}_{ey}",
                    {
                        "domain": "europe",                                         # fixed
                        "experiment": ["historical", "rcp_4_5"],                    # change with rcp_8_5
                        "horizontal_resolution": "0_11_degree_x_0_11_degree",       # fixed
                        "temporal_resolution": "daily_mean",                        # fixed
                        "variable": variable,                                       # Argument of the function
                        "gcm_model": gcm,                                           # fixed
                        "rcm_model": rcm,                                           # fixed
                        "ensemble_member": ensemble_model,                          # fixed
                        "start_year": sy,                                           # Single start year
                        "end_year": ey,                                             # Single end year
                    },
                ),
            ]

            download_dfs(request_models, dfs_chunks)                  # Download of the dictionary
        dfs_models[model] = dfs_chunks                                # Fill the main dictionary
    return dfs_models

def concat_models_along_time(dfs_models):
    """
    Concatenate the temporal chunks of each model along the 'time' dimension.

    Parameters
    ----------
    dfs_models : dict
        A dictionary where each key represents a model name and each value 
        contains multiple xarray.Dataset objects corresponding to different 
        time chunks.

    Returns
    -------
    dict
        A dictionary mapping each model name to a single xarray.Dataset
        obtained by concatenating all its time chunks along the 'time' dimension.
    """
    dfs_models_concat = {}                             # Main dictionary

    for model, chunks in dfs_models.items():           # Loop over each model
        keys_sorted = sorted(chunks.keys())            # Ensure chronological order
        ds_list = [chunks[k] for k in keys_sorted]     # Extract datasets in order
        ds_concat = xr.concat(ds_list, dim="time")     # Concatenate along time
        dfs_models_concat[model] = ds_concat           # Fill the main dictionary
    return dfs_models_concat

def split_time_windows(dfs_yearly_ext, window_length=40, shift=10):
    """
    Split yearly data into sliding time windows for each model.

    For every model dataset:
    - Extract the available years from the 'time' coordinate.
    - Generate time windows of length `window_length` years, moving forward
      by `shift` years between consecutive windows.
    - For each window, select the subset of data whose years fall within
      the window range (start_year --> end_year).
    - Store each windowed dataset in a dictionary indexed by (start_year, end_year).

    The function returns a dictionary where each model is associated with
    multiple time-window datasets, useful for analyzing long-term climate
    extremes over overlapping periods.
    """
    dfs_window = {}                                             # Main dictionary with data divided by time windows

    for model, ds in dfs_yearly_ext.items():                    # Loop over each model
        years = ds["time"].dt.year.values
        min_year, max_year = int(years.min()), int(years.max())
        windows = [
            (start, start + window_length - 1)                  # For each start (year), it generates a time window "window_length - 1" long.
            for start in range(min_year, max_year + 1, shift)
            if start + window_length - 1 <= max_year            # Exclude time windows that end beyond max_year.
        ]
        windowed_data = {}                                      # Fill the dictionary with only time windows (?)

        for start, end in windows:
            mask = (ds["time"].dt.year >= start) & (ds["time"].dt.year <= end)
            ds_window = ds.sel(time=mask)
            windowed_data[(start, end)] = ds_window
            print(f"model: {model}, window: {start}/{end}, shape: {dict(ds_window.sizes)}")
        dfs_window[model] = windowed_data

    return dfs_window

def new_ev1_param_cerra(data, T_r=50, method='LSM', value='positive'):
    original_data = data
    data = np.asarray(data).astype(float)
    p = 1 / T_r
    yp = -np.log(-np.log(1 - p))
    N = data.shape[0]
    pp = np.arange(N, 0, -1) / (N + 1)
    loglog_pp = np.log(-np.log(pp)).reshape(N, 1, 1)

    if value == 'positive':
        data_sorted = np.sort(data, axis=0)[::-1]

        # Solo valori strettamente > 0 per i parametri (NaN e zeri esclusi).
        # N e loglog_pp (plotting position) restano calcolati su TUTTI gli anni.
        valid = data_sorted > 0
        n = valid.sum(axis=0)
        n_safe = np.where(n >= 3, n, np.nan)   # soglia minima di anni con neve
        x  = np.where(valid, data_sorted, 0.0)
        ll = np.where(valid, loglog_pp, 0.0)

        if method == 'LSM':
            parmhat_2 = (
                (np.sum(x, axis=0)**2 - n_safe * np.sum(x**2, axis=0)) /
                (n_safe * np.sum(x * ll, axis=0)
                 - np.sum(x, axis=0) * np.sum(ll, axis=0)))
            parmhat_1 = (np.sum(x, axis=0) / n_safe
                + np.sum(ll, axis=0) * parmhat_2 / n_safe)

        elif method == 'MOM':
            mean_val = np.sum(x, axis=0) / n_safe
            std_val = np.sqrt((np.sum(x**2, axis=0) - n_safe * mean_val**2) / (n_safe - 1))
            parmhat_2 = np.sqrt(6) / np.pi * std_val
            parmhat_1 = mean_val - 0.5772157 * parmhat_2


    elif value == 'negative':

        if method == 'MOM':
            data_sorted = np.sort(data, axis=0)[::-1]
            mean_val = np.mean(data_sorted, axis=0)
            std_val = np.std(data_sorted, ddof=1, axis=0)
            parmhat_2 = -np.sqrt(6) / np.pi * std_val
            parmhat_1 = mean_val - 0.5772157 * (-parmhat_2)

        elif method == 'LSM':
            data_sorted = np.sort(data, axis=0)
            parmhat_2 = ((np.sum(data_sorted, axis=0)**2 - N * np.sum(data_sorted**2, axis=0)) /
                (N * np.sum(data_sorted * loglog_pp, axis=0)
                 - np.sum(data_sorted, axis=0) * np.sum(loglog_pp, axis=0)))
            parmhat_1 = (np.mean(data_sorted, axis=0) + np.sum(loglog_pp * parmhat_2, axis=0) / N)

    R_value = parmhat_1 + parmhat_2 * yp

    new_dict = {
        'latitude': (('y', 'x'), original_data['latitude'].values),
        'longitude': (('y', 'x'), original_data['longitude'].values),
    }
    return_level = xr.DataArray(data=R_value, dims=['y', 'x'], coords=new_dict)
    location = xr.DataArray(data=parmhat_1, dims=['y', 'x'], coords=new_dict)
    scale = xr.DataArray(data=parmhat_2, dims=['y', 'x'], coords=new_dict)
    return return_level, location, scale

def compute_window_results(dfs_window, T_r=50, variable="tasmax", method='MOM'):
    """
    Compute the temporal mean for each model and
    each defined time window.
    """
    dfs_window_means = {}
    dfs_window_cov = {}
    dfs_window_charact = {}

    positive_vars = {"tasmax", "pr", "sfcWind"}

    for model, windows in dfs_window.items():

        window_means = {}
        window_cov = {}
        window_charact = {}

        for (start, end), ds_window in windows.items():

            # Estrai la variabile specifica
            da_window = ds_window[variable]

            # Mean value
            window_means[(start, end)] = da_window.mean(dim="time")

            # COV
            window_cov[(start, end)] = (
                da_window.std(dim="time") / da_window.mean(dim="time")
            )

            # Characteristic value (EV1)
            if variable in positive_vars:
                value_type = "positive"
            elif variable == "tasmin":
                    value_type = "negative"

            window_charact[(start, end)] = new_ev1_param(
                da_window,
                T_r=T_r,
                method=method,
                value=value_type
            )

        dfs_window_means[model] = window_means
        dfs_window_cov[model] = window_cov
        dfs_window_charact[model] = window_charact

    return dfs_window_means, dfs_window_cov, dfs_window_charact

def compute_fc_from_charact(dfs_charact, variable):
    """
    Compute the factor of change (fc) from dfs_charact.

    For:
    - tasmax, tasmin: fc = difference with respect to the first time window
    - pr, sfcWind:   fc = ratio with respect to the first time window

    Parameters
    ----------
    dfs_charact : dict
        Dictionary structured as dfs_charact[model][(start, end)] = xr.DataArray
    variable: str
        Climate variable name ('tasmax', 'tasmin', 'pr', 'sfcWind')

    Returns
    -------
    dfs_fc : dict
        Dictionary with the same structure as dfs_charact containing fc values
    """

    if variable not in ["tasmax", "tasmin", "pr", "sfcWind"]:
        raise ValueError(f"Variable '{variable}' not supported for fc computation")

    dfs_fc = {}

    for model, windows in dfs_charact.items():

        sorted_windows = sorted(windows.keys(), key=lambda x: x[0])

        ref_window = sorted_windows[0]
        ref_data = windows[ref_window]

        model_fc = {}

        for window in sorted_windows:
            current_data = windows[window]

            if variable in ["tasmax", "tasmin"]:
                # difference
                fc = current_data - ref_data

            elif variable in ["pr", "sfcWind"]:
                # division
                fc = current_data / ref_data

            model_fc[window] = fc

        dfs_fc[model] = model_fc

    return dfs_fc

def extract_results(dfs_results, lat, lon):
    """
    Finds the closest (rlat,rlon) values to the given (lat, lon) for each model 
    and each available time window (period) in dfs_results. The function first converts 
    the input geographic coordinates into the corresponding model grid coordinates,
    and only after that extracts the matching dataframes.
    """

    coords = {}
    result_dfs = {}

    for key in dfs_results:
        coords[key] = {}
        result_dfs[key] = {}

        for period in dfs_results[key].keys():
            dfs = dfs_results[key][period]

            dist = (dfs.lat - lat)**2 + (dfs.lon - lon)**2
            idx = dist.argmin(dim=("rlat", "rlon"))

            rlat_idx = idx["rlat"]
            rlon_idx = idx["rlon"]

            rlat_value = dfs.rlat.isel(rlat=rlat_idx).item()
            rlon_value = dfs.rlon.isel(rlon=rlon_idx).item()

            coords[key] = (rlat_value, rlon_value)

            result_dfs[key][period] = dfs.sel(
                rlat=rlat_value,
                rlon=rlon_value,
                method="nearest"
            )


    return coords, result_dfs

def export_csv(dfs, label, out_filename):
    """
    Save all the data in a .csv file, and export it.
    """

    all_models = []

    for model, windows in dfs.items():
        print("Processing model:", model)

        df_model = None

        for (start, end), da in windows.items():
            da = da.compute()

            lat2d  = da["lat"].values
            lon2d  = da["lon"].values
            rlat1d = da["rlat"].values
            rlon1d = da["rlon"].values
            vals   = da.values

            nlat, nlon = vals.shape
            records = []

            for i in range(nlat):
                for j in range(nlon):
                    v = vals[i, j]
                    if np.isnan(v):
                        continue

                    records.append({
                        "lat":  float(lat2d[i, j]),
                        "lon":  float(lon2d[i, j]),
                        "rlat": float(rlat1d[i]),
                        "rlon": float(rlon1d[j]),
                        f"{label}_{start}_{end}": float(v)
                    })

            df_window = pd.DataFrame.from_records(records)

            if df_model is None:
                df_model = df_window
            else:
                df_model = df_model.merge(
                    df_window,
                    on=["lat","lon","rlat","rlon"],
                    how="outer"
                )

        df_model.insert(0, "model", model)

        all_models.append(df_model)

    final_df = pd.concat(all_models, ignore_index=True)
    final_df.to_csv(
        out_filename,
        index=False,
        sep=";",
        decimal=",",
        float_format="%.6f"
    )

    print("CSV saved:", len(final_df), "rows ->", out_filename)

#### Import Orography

In [4]:
# Cartella contenente tutti i NetCDF
path_orog = Path(r"D:\DOTTORATO\21_CERRA\0_cerra_dati")

# Tutti i file di snow_density
files_orog = sorted(path_orog.glob("cerra_orography*.nc"))

# Apre e unisce automaticamente lungo la coordinata temporale
orog = xr.open_mfdataset(files_orog,combine="by_coords")

bbox = (
    (orog["latitude"].values >= 36) & (orog["latitude"].values <= 47) &
    (orog["longitude"].values >= 6) & (orog["longitude"].values <= 19)
)
iy, ix = np.where(bbox)
orog = orog.isel(
    y=slice(iy.min(), iy.max() + 1),
    x=slice(ix.min(), ix.max() + 1)
)
orog

<xarray.Dataset> Size: 1MB
Dimensions:     (valid_time: 1, y: 236, x: 218)
Coordinates:
  * valid_time  (valid_time) datetime64[ns] 8B 1984-09-01
    latitude    (y, x) float64 412kB dask.array<chunksize=(236, 218), meta=np.ndarray>
    longitude   (y, x) float64 412kB dask.array<chunksize=(236, 218), meta=np.ndarray>
    expver      <U4 16B ...
Dimensions without coordinates: y, x
Data variables:
    orog        (valid_time, y, x) float32 206kB dask.array<chunksize=(1, 236, 218), meta=np.ndarray>
Attributes:
    GRIB_centre:             eswi
    GRIB_centreDescription:  Norrkoping
    GRIB_subCentre:          255
    Conventions:             CF-1.7
    institution:             Norrkoping
    history:                 2026-11-16T10:54 GRIB to CDM+CF via cfgrib-0.9.1...

#### Import data cerra (Already locally downloaded)

In [5]:
# Cartella contenente tutti i NetCDF
path = Path(r"D:\DOTTORATO\16_GITHUB\HBM-ABCSubSim\CERRA sub-daily reanalysis copernicus\CERRA data\test")

# Tutti i file di snow_density
files = sorted(path.glob("SDWE_cerra_*.nc"))

# Apre e unisce automaticamente lungo la coordinata temporale
df = xr.open_mfdataset(files,combine="by_coords")

bbox = (
    (df["latitude"].values >= 36) & (df["latitude"].values <= 47) &
    (df["longitude"].values >= 6) & (df["longitude"].values <= 19)
)
iy, ix = np.where(bbox)
df = df.isel(
    y=slice(iy.min(), iy.max() + 1),
    x=slice(ix.min(), ix.max() + 1)
)
df

<xarray.Dataset> Size: 226MB
Dimensions:     (valid_time: 1095, y: 236, x: 218)
Coordinates:
  * valid_time  (valid_time) datetime64[ns] 9kB 1985-01-01 ... 1987-12-31
    latitude    (y, x) float64 412kB dask.array<chunksize=(236, 218), meta=np.ndarray>
    longitude   (y, x) float64 412kB dask.array<chunksize=(236, 218), meta=np.ndarray>
    expver      (valid_time) <U4 18kB dask.array<chunksize=(365,), meta=np.ndarray>
Dimensions without coordinates: y, x
Data variables:
    sd          (valid_time, y, x) float32 225MB dask.array<chunksize=(73, 179, 141), meta=np.ndarray>
Attributes:
    GRIB_centre:             eswi
    GRIB_centreDescription:  Norrkoping
    GRIB_subCentre:          255
    Conventions:             CF-1.7
    institution:             Norrkoping
    history:                 2026-11-13T14:42 GRIB to CDM+CF via cfgrib-0.9.1...

In [6]:
# ------------------------------------------------------------
# Lettura e preparazione dello shapefile
# ------------------------------------------------------------

shapefile_path = (
    r"C:\Users\Costanza\Downloads\Limiti01012026"
    r"\ProvCM01012026\ProvCM01012026_WGS84.shp"
)

shp = gpd.read_file(shapefile_path)

if shp.crs is None:
    shp = shp.set_crs("EPSG:4326")
else:
    shp = shp.to_crs("EPSG:4326")

boundary_union = shp.geometry.union_all()

# ------------------------------------------------------------
# Coordinate 2D della griglia ERA5
# ------------------------------------------------------------

lat = df["latitude"].values
lon = df["longitude"].values

# ------------------------------------------------------------
# Creazione maschera spaziale
# Elaborazione per blocchi di righe
# ------------------------------------------------------------

inside_mask = np.zeros(lat.shape, dtype=bool)

block_size = 100

for i in range(0, lat.shape[0], block_size):

    i_end = min(i + block_size, lat.shape[0])

    inside_mask[i:i_end, :] = contains_xy(
        boundary_union,
        lon[i:i_end, :],
        lat[i:i_end, :]
    )

# ------------------------------------------------------------
# Creazione DataArray della maschera
# ------------------------------------------------------------

mask = xr.DataArray(
    inside_mask,
    coords={
        "y": df["y"],
        "x": df["x"]
    },
    dims=("y", "x")
)

# ------------------------------------------------------------
# Filtraggio spaziale
# ------------------------------------------------------------

df_italy = df.where(mask, drop=True)

# ------------------------------------------------------------
# Estrazione SWE
# ------------------------------------------------------------

# df_swe = (
#     df_italy["sd"]
#     .to_dataframe()
#     .reset_index()
#     .rename(columns={
#         "sd": "sdwe"
#     })
# )

# print("Shape SWE:", df_swe.shape)


In [7]:
# SDWE [kg/m²], escludendo le celle glaciali (soglia da calibrare)
sd = df_italy["sd"]
sd = sd.where(sd <= 1000)

# Orografia sulla stessa griglia (236x218, la carico in memoria)
orog_2d = (
     orog["orog"].isel(valid_time=0, drop=True)
     .drop_vars("expver", errors="ignore")
     .assign_coords(y=orog["y"], x=orog["x"])
     .reindex_like(sd.isel(valid_time=0, drop=True))
     .load())

assert np.allclose(orog_2d["latitude"].values, sd["latitude"].values)
assert np.allclose(orog_2d["longitude"].values, sd["longitude"].values)

# Maschera di quota applicata PRIMA del massimo stagionale
sd = sd.where(orog_2d <= 1500)
sd = sd.assign_coords(altitude=orog_2d)

# SDWE [kg/m²] -> snow load [kN/m²]
snow_load = sd * 9.81 / 1000

# Solo novembre-aprile
month = snow_load["valid_time"].dt.month
snow_load = snow_load.sel(valid_time=(month >= 11) | (month <= 4))

# Anno di inizio della stagione invernale (gen-apr -> anno precedente)
year = snow_load["valid_time"].dt.year
month = snow_load["valid_time"].dt.month
snow_load = snow_load.assign_coords(
    year=("valid_time", (year - (month <= 4).astype(int)).values)
)

# Massimo stagionale per ogni cella
seasonal_max_xr = snow_load.groupby("year").max("valid_time").compute()

df_swe = (
    seasonal_max_xr
    .drop_vars("expver", errors="ignore")
    .to_dataframe(name="snow_load_max")
    .reset_index()
)

print("Shape SWE:", df_swe.shape)

Shape SWE: (155296, 7)


#### Only Points Inside Italy

In [8]:
# ------------------------------------------------------------------
# Keep only the CERRA grid points located inside Italy
# ------------------------------------------------------------------

shapefile_path = r"C:\Users\Costanza\Downloads\Limiti01012026\ProvCM01012026\ProvCM01012026_WGS84.shp"

shp = gpd.read_file(shapefile_path)

if shp.crs is None:
    shp = shp.set_crs("EPSG:4326")
elif shp.crs != "EPSG:4326":
    shp = shp.to_crs("EPSG:4326")

# Extract unique grid points (the grid is identical for depth and density)
unique_points = (
    df_swe[["latitude", "longitude"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

points_gdf = gpd.GeoDataFrame(
    unique_points,
    geometry=gpd.points_from_xy(
        unique_points["longitude"],
        unique_points["latitude"]
    ),
    crs="EPSG:4326"
)

boundary_union = shp.geometry.union_all()

inside_points = (
    points_gdf[points_gdf.geometry.within(boundary_union)]
    [["latitude", "longitude"]]
    .copy()
)

# print(f"Grid points inside Italy: {len(inside_points)}")

# Filtra snow depth
df_table = df_swe.merge(
    inside_points,
    on=["latitude", "longitude"],
    how="inner"
)

df_table = df_table[df_table["altitude"] <= 1500]

#### Shapefile

In [9]:
# # Read shapefile
# shp = gpd.read_file(shapefile_path)

# # Ensure the shapefile is in WGS84 / EPSG:4326
# if shp.crs is None:
#     shp = shp.set_crs("EPSG:4326")
# elif shp.crs != "EPSG:4326":
#     shp = shp.to_crs("EPSG:4326")

# # Keep only unique latitude/longitude pairs once
# unique_points = (
#     df_table[["latitude", "longitude"]]
#     .drop_duplicates()
#     .reset_index(drop=True)
# )

# # Build a GeoDataFrame with the unique points
# points_gdf = gpd.GeoDataFrame(
#     unique_points,
#     geometry=gpd.points_from_xy(unique_points["longitude"], unique_points["latitude"]),
#     crs="EPSG:4326"
# )

# # Keep only points that fall inside the shapefile boundary
# boundary_union = shp.geometry.unary_union
# inside_points = points_gdf[points_gdf.geometry.within(boundary_union)].copy()

# print("Unique points inside shapefile:", len(inside_points))

# # Plot with Plotly: one color for all points, one legend entry
# fig = px.scatter_mapbox(
#     inside_points,
#     lat="latitude",
#     lon="longitude",
#     color_discrete_sequence=["#1f77b4"],
#     size_max=15,
#     hover_data={"latitude": True, "longitude": True},
#     zoom=5,
#     height=700
# )

# # Add the shapefile boundary as a single black outline trace
# boundary_geom = shp.geometry.unary_union

# if boundary_geom.geom_type == "Polygon":
#     coords = np.array(boundary_geom.exterior.coords)
#     fig.add_trace(
#         go.Scattermapbox(
#             lat=coords[:, 1],
#             lon=coords[:, 0],
#             mode="lines",
#             line=dict(color="black", width=2),
#             name="Shapefile boundary",
#             hoverinfo="skip",
#             showlegend=False
#         )
#     )
# elif boundary_geom.geom_type == "MultiPolygon":
#     for poly in boundary_geom.geoms:
#         coords = np.array(poly.exterior.coords)
#         fig.add_trace(
#             go.Scattermapbox(
#                 lat=coords[:, 1],
#                 lon=coords[:, 0],
#                 mode="lines",
#                 line=dict(color="black", width=2),
#                 name="Shapefile boundary",
#                 hoverinfo="skip",
#                 showlegend=False
#             )
#         )

# fig.update_layout(
#     mapbox_style="white-bg",
#     margin={"r": 0, "t": 0, "l": 0, "b": 0},
#     title="Unique grid points inside the shapefile boundaries"
# )

# fig.show()

#### Seasonal Maximum

In [10]:
seasonal_max = df_table[["year", "latitude", "longitude", "snow_load_max"]].copy()

#### EVA

In [11]:
# Massimi stagionali (year, y, x), senza la stagione parziale 1984
data_ev = seasonal_max_xr.isel(year=(seasonal_max_xr["year"] > 1984).values)

data_ev = data_ev.transpose("year", "y", "x")

R50, location, scale = new_ev1_param_cerra(data_ev, T_r=50, method='LSM')

print("Celle con valore:", int(np.isfinite(R50).sum()), "su", int(np.isfinite(data_ev).any("year").sum()))

Celle con valore: 2022 su 9378


#### Semivariogrammi

Estimator	|Caratteristica	                                          |Quando utilizzarlo

Matheron	| Media delle differenze quadratiche	                  | Dati senza outlier importanti

Cressie	    | Stimatore robusto                                       | Presenza di valori anomali

Dowd	    | Basato sulle differenze assolute e sulla mediana	      | Dati con distribuzione asimmetrica o outlier

Genton	    | Stimatore robusto basato sulla mediana delle differenze | Maggiore robustezza rispetto a valori estremi

In [12]:
# ------------------------------------------------------------
# SEMIVARIOGRAMMA DEI PARAMETRI GUMBEL (mu, sigma, log sigma)
# ------------------------------------------------------------

# ---------------- SCELTE ----------------
param     = "mu"           # mu | sigma | logsigma
estimator = "matheron"     # matheron | cressie | dowd | genton
model     = "gaussian"  # exponential | gaussian | spherical
nlags     = 30
maxlag_km = 300          # es. 600 per limitare la distanza massima; None = default skgstat
max_points = 6000          # sottocampionamento se i punti sono di più (None = usa tutti)
seed = 42
# ----------------------------------------

# Tabella dei parametri sulla griglia, solo dentro l'Italia
df_par = pd.DataFrame({
    "latitude":  location["latitude"].values.ravel(),
    "longitude": location["longitude"].values.ravel(),
    "mu":        location.values.ravel(),
    "sigma":     scale.values.ravel(),
})

df_par = df_par.merge(
    inside_points[["latitude", "longitude"]],
    on=["latitude", "longitude"],
    how="inner"
)

# Variabile scelta
if param == "mu":
    df_par["value"] = df_par["mu"]
    label = "μ (location)"
elif param == "sigma":
    df_par["value"] = df_par["sigma"]
    label = "σ (scale)"
elif param == "logsigma":
    df_par = df_par[df_par["sigma"] > 0].copy()      # il log richiede sigma > 0
    df_par["value"] = np.log(df_par["sigma"])
    label = "log σ"
else:
    raise ValueError("param deve essere 'mu', 'sigma' o 'logsigma'")

df_par = df_par[np.isfinite(df_par["value"])].reset_index(drop=True)

# Sottocampionamento opzionale (le distanze a coppie crescono con N^2)
if max_points is not None and len(df_par) > max_points:
    df_par = df_par.sample(max_points, random_state=seed).reset_index(drop=True)

print(f"Punti usati per {label}: {len(df_par)}")

# ------------------------------------------------------------
# Coordinate metriche (stessa proiezione che avevi già)
# ------------------------------------------------------------
lats = df_par["latitude"].to_numpy(dtype=float)
lons = df_par["longitude"].to_numpy(dtype=float)
vals = df_par["value"].to_numpy(dtype=float)

R_earth = 6371000
lat0 = np.mean(lats)
lon0 = np.mean(lons)

x_coords = R_earth * np.cos(np.radians(lat0)) * np.radians(lons - lon0)
y_coords = R_earth * np.radians(lats - lat0)
coords_par = np.column_stack((x_coords, y_coords))

# ------------------------------------------------------------
# Variogramma
# ------------------------------------------------------------
vario_kwargs = dict(
    n_lags=nlags,
    use_nugget=True,
    estimator=estimator,
    model=model,
)
if maxlag_km is not None:
    vario_kwargs["maxlag"] = maxlag_km * 1000

semivar = Variogram(coords_par, vals, **vario_kwargs)

# Numero di coppie per ogni classe di distanza
pair_counts = np.array([
    len(pairs) for pairs in semivar.lag_classes()
])

print("\nNumero di coppie per lag:")
for i, (h, n) in enumerate(zip(semivar.bins, pair_counts)):
    print(f"Lag {i+1:3d} | {h/1000:7.1f} km | {n:10d} coppie")

print(semivar)
range_, sill_, nugget_ = semivar.parameters[:3]
print(f"Range: {range_/1000:.1f} km | Sill: {sill_:.4g} | Nugget: {nugget_:.4g}")

# ------------------------------------------------------------
# Plot con Plotly
# ------------------------------------------------------------
lags = np.asarray(semivar.bins)
gamma_exp = np.asarray(semivar.experimental)

distances = np.linspace(0, np.nanmax(lags), 300)
gamma_model = semivar.fitted_model(distances)

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=lags / 1000,
    y=gamma_exp,
    mode="markers",
    name=f"Experimental ({estimator})",
    marker=dict(size=9, color="royalblue", symbol="circle",
                line=dict(width=1, color="black"))
))

fig.add_trace(go.Scatter(
    x=distances / 1000,
    y=gamma_model,
    mode="lines",
    name=f"{model.capitalize()} model",
    line=dict(color="firebrick", width=2.5)
))

# Sill e nugget come riferimento
fig.add_hline(y=sill_ + nugget_ if False else sill_, line=dict(color="grey", dash="dash"),
              annotation_text=f"sill = {sill_:.3g}", annotation_position="top right")

fig.update_layout(
    title=dict(text=f"Semivariogram of {label} – Italy",
               x=0.5, xanchor="center"),
    xaxis_title="Lag distance [km]",
    yaxis_title="Semivariance",
    template="plotly_white",
    width=950,
    height=600,
    font=dict(family="Arial", size=13),
    legend=dict(x=0.02, y=0.98, bgcolor="rgba(255,255,255,0.8)"),
    margin=dict(l=80, r=40, t=80, b=70)
)
fig.update_xaxes(showgrid=True, gridcolor="lightgrey")
fig.update_yaxes(showgrid=True, gridcolor="lightgrey")

fig.show()

Punti usati per μ (location): 2022

Numero di coppie per lag:
Lag   1 |    10.0 km |       6159 coppie
Lag   2 |    20.0 km |      21681 coppie
Lag   3 |    30.0 km |      27739 coppie
Lag   4 |    40.0 km |      33756 coppie
Lag   5 |    50.0 km |      32789 coppie
Lag   6 |    60.0 km |      35205 coppie
Lag   7 |    70.0 km |      35969 coppie
Lag   8 |    80.0 km |      35781 coppie
Lag   9 |    90.0 km |      33593 coppie
Lag  10 |   100.0 km |      32144 coppie
Lag  11 |   110.0 km |      30431 coppie
Lag  12 |   120.0 km |      31033 coppie
Lag  13 |   130.0 km |      32734 coppie
Lag  14 |   140.0 km |      34809 coppie
Lag  15 |   150.0 km |      36510 coppie
Lag  16 |   160.0 km |      37232 coppie
Lag  17 |   170.0 km |      37845 coppie
Lag  18 |   180.0 km |      38822 coppie
Lag  19 |   190.0 km |      40331 coppie
Lag  20 |   200.0 km |      39079 coppie
Lag  21 |   210.0 km |      38795 coppie
Lag  22 |   220.0 km |      39150 coppie
Lag  23 |   230.0 km |      37845 co

In [13]:
# ------------------------------------------------------------
# CONFRONTO MODELLI TEORICI DEL SEMIVARIOGRAMMA
# RMSE rispetto ai punti sperimentali
# ------------------------------------------------------------

# ------------------------------------------------------------
# IMPOSTAZIONI
# ------------------------------------------------------------

estimators = [
    "matheron",
    "cressie",
    "dowd"
]

models = [
    "spherical",
    "exponential",
    "gaussian"
]

maxlag_km = 300
n_lags = 30

maxlag = maxlag_km * 1000

# Coordinate e valori già definiti:
# coords_par
# vals

# ------------------------------------------------------------
# CALCOLO VARIOGRAMMI E RMSE
# ------------------------------------------------------------

results = []
variograms = {}

for estimator in estimators:

    for model in models:

        try:

            V = Variogram(
                coords_par,
                vals,
                estimator=estimator,
                model=model,
                maxlag=maxlag,
                n_lags=n_lags,
                use_nugget=True
            )

            # --------------------------------------------
            # PUNTI SPERIMENTALI
            # --------------------------------------------

            distances = np.asarray(V.bins, dtype=float)
            gamma_exp = np.asarray(V.experimental, dtype=float)

            # Elimina eventuali lag non validi
            valid = (
                np.isfinite(distances) &
                np.isfinite(gamma_exp)
            )

            distances = distances[valid]
            gamma_exp = gamma_exp[valid]

            # --------------------------------------------
            # MODELLO TEORICO
            # --------------------------------------------

            gamma_fit = np.asarray(
                V.fitted_model(distances),
                dtype=float
            )

            valid_fit = (
                np.isfinite(gamma_exp) &
                np.isfinite(gamma_fit)
            )

            gamma_exp_valid = gamma_exp[valid_fit]
            gamma_fit_valid = gamma_fit[valid_fit]

            # --------------------------------------------
            # RMSE
            # --------------------------------------------

            if len(gamma_exp_valid) > 0:

                rmse = np.sqrt(
                    np.mean(
                        (gamma_exp_valid - gamma_fit_valid) ** 2
                    )
                )

            else:
                rmse = np.nan

            # --------------------------------------------
            # PARAMETRI DEL MODELLO
            # --------------------------------------------

            range_, sill_, nugget_ = V.parameters[:3]

            results.append({
                "Estimator": estimator,
                "Model": model,
                "Range (km)": range_ / 1000,
                "Sill": sill_,
                "Nugget": nugget_,
                "RMSE": rmse,
                "N. lag": len(gamma_exp_valid),
                "Status": "OK"
            })

            variograms[(estimator, model)] = {
                "variogram": V,
                "distances": distances,
                "experimental": gamma_exp,
                "fitted": gamma_fit
            }

        except Exception as e:

            results.append({
                "Estimator": estimator,
                "Model": model,
                "Range (km)": np.nan,
                "Sill": np.nan,
                "Nugget": np.nan,
                "RMSE": np.nan,
                "N. lag": 0,
                "Status": f"Errore: {str(e)[:100]}"
            })

# ------------------------------------------------------------
# TABELLA RISULTATI
# ------------------------------------------------------------

df_results = pd.DataFrame(results)

df_results = df_results.sort_values(
    by="RMSE",
    ascending=True,
    na_position="last"
).reset_index(drop=True)

print("\nCONFRONTO MODELLI - RMSE")
display(
    df_results.style.format({
        "Range (km)": "{:.1f}",
        "Sill": "{:.5f}",
        "Nugget": "{:.5f}",
        "RMSE": "{:.5f}"
    })
)

# ------------------------------------------------------------
# MIGLIORE COMBINAZIONE
# ------------------------------------------------------------

df_valid = df_results[
    df_results["RMSE"].notna()
]

if not df_valid.empty:

    best = df_valid.iloc[0]

    print("\nMIGLIORE COMBINAZIONE SECONDO RMSE")
    print(f"Estimator : {best['Estimator']}")
    print(f"Model     : {best['Model']}")
    print(f"RMSE      : {best['RMSE']:.5f}")
    print(f"Range     : {best['Range (km)']:.1f} km")
    print(f"Sill      : {best['Sill']:.5f}")
    print(f"Nugget    : {best['Nugget']:.5f}")

# ------------------------------------------------------------
# GRAFICI: PUNTI SPERIMENTALI E MODELLI TEORICI
# Un grafico per ogni stimatore
# ------------------------------------------------------------

for estimator in estimators:

    fig = go.Figure()

    plotted = False

    for model in models:

        key = (estimator, model)

        if key not in variograms:
            continue

        data = variograms[key]

        distances_km = data["distances"] / 1000
        gamma_exp = data["experimental"]

        # Punti sperimentali: visualizzati una sola volta
        if not plotted:

            fig.add_trace(go.Scatter(
                x=distances_km,
                y=gamma_exp,
                mode="markers",
                name="Experimental",
                marker=dict(size=7, color="black")
            ))

            plotted = True

        # Curva teorica su intervallo continuo
        V = data["variogram"]

        x_fit = np.linspace(0, maxlag, 300)

        y_fit = V.fitted_model(x_fit)

        fig.add_trace(go.Scatter(
            x=x_fit / 1000,
            y=y_fit,
            mode="lines",
            name=model.capitalize()
        ))

    fig.update_layout(
        title=f"Semivariogram - {estimator.capitalize()}",
        xaxis_title="Distance (km)",
        yaxis_title="Semivariance",
        template="plotly_white",
        font=dict(family="Arial"),
        width=850,
        height=550,
        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=1.02,
            xanchor="left",
            x=0
        )
    )

    fig.show()


CONFRONTO MODELLI - RMSE


,Estimator,Model,Range (km),Sill,Nugget,RMSE,N. lag,Status
0,cressie,exponential,106.6,0.10591,0.00381,0.00472,30,OK
1,cressie,spherical,92.3,0.08879,0.01887,0.00539,30,OK
2,cressie,gaussian,89.4,0.07631,0.03135,0.00548,30,OK
3,dowd,gaussian,75.6,0.06000,0.00920,0.00607,30,OK
4,dowd,spherical,78.0,0.06916,0.00000,0.00608,30,OK
5,dowd,exponential,109.2,0.07112,0.00000,0.00621,30,OK
6,matheron,spherical,111.0,0.10256,0.09654,0.00694,30,OK
7,matheron,gaussian,106.9,0.08831,0.11071,0.00743,30,OK
8,matheron,exponential,106.4,0.13145,0.06855,0.00780,30,OK



MIGLIORE COMBINAZIONE SECONDO RMSE
Estimator : cressie
Model     : exponential
RMSE      : 0.00472
Range     : 106.6 km
Sill      : 0.10591
Nugget    : 0.00381


In [14]:
# df_recap = (
#     seasonal_max
#     .groupby(["latitude", "longitude"], as_index=False)
#     .agg(
#         Mean=("snow_load_max", "mean"),
#         StdDvz=("snow_load_max", "std"),
#         Variance=("snow_load_max", "var"),
#         N=("snow_load_max", "count"),
#     )
# )

# #############################################################################################################################################
# # Individua le celle con deviazione standard nulla
# mask_zero_std = df_recap["StdDvz"] == 0

# n_zero_std = mask_zero_std.sum()

# if n_zero_std > 0:
#     print(
#         f"ATTENZIONE: {n_zero_std} celle hanno tutti i massimi stagionali uguali "
#         "→ StdDvz = 0 → sostituisco con NaN."
#     )

#     # Sostituisce la deviazione standard nulla con NaN
#     df_recap.loc[mask_zero_std, "StdDvz"] = np.nan

#     # Anche la varianza deve diventare NaN
#     df_recap.loc[mask_zero_std, "Variance"] = np.nan
# # Caso speciale perchè ho pochi dati
# #############################################################################################################################################

# means = df_recap["Mean"].to_numpy()
# stds  = df_recap["StdDvz"].to_numpy()

# lats = df_recap["latitude"].to_numpy(dtype=float)
# lons = df_recap["longitude"].to_numpy(dtype=float)

# R = 6371000  # m

# lat0 = np.mean(lats)
# lon0 = np.mean(lons)

# x_coords = R * np.cos(np.radians(lat0)) * np.radians(lons - lon0)
# y_coords = R * np.radians(lats - lat0)

# coords = np.column_stack((x_coords, y_coords))

In [15]:
# # ------------------------------------------------------------
# # VARIOGRAM OF MEAN SEASONAL SNOW LOAD
# # ------------------------------------------------------------

# valid_mu = (
#     df_recap["Mean"].notna() &
#     np.isfinite(df_recap["Mean"])
# )

# df_mu = df_recap.loc[valid_mu].copy()

# lats = df_mu["latitude"].to_numpy(dtype=float)
# lons = df_mu["longitude"].to_numpy(dtype=float)

# means = df_mu["Mean"].to_numpy(dtype=float)

# R = 6371000

# lat0 = np.mean(lats)
# lon0 = np.mean(lons)

# x_coords = (
#     R * np.cos(np.radians(lat0)) *
#     np.radians(lons - lon0)
# )

# y_coords = R * np.radians(lats - lat0)

# coords_mu = np.column_stack((x_coords, y_coords))

# # ------------------------------------------------------------
# # VARIOGRAM
# # ------------------------------------------------------------

# nlags = 100

# semivar_model = "exponential"   # exponential | Gaussian, Spherical, Linear, Power, Stable, Nugget

# semivar_mu = Variogram(
#     coords_mu,
#     means,
#     n_lags=nlags,
#     use_nugget=True,
# )

# semivar_mu.model = semivar_model

# print(semivar_mu)

In [16]:
# # ------------------------------------------------------------
# # PLOT SEMIVARIOGRAMMA CON PLOTLY
# # ------------------------------------------------------------

# import plotly.graph_objects as go
# import numpy as np

# # Experimental variogram
# lags = semivar_mu.bins
# gamma_exp = semivar_mu.experimental

# # Theoretical variogram
# max_lag = np.nanmax(lags)
# distances = np.linspace(0, max_lag, 300)
# gamma_theoretical = semivar_mu.transform(distances)

# # Model parameters
# nugget = semivar_mu.parameters[2] if semivar_mu.use_nugget else 0

# # ------------------------------------------------------------
# # FIGURE
# # ------------------------------------------------------------

# fig = go.Figure()

# # Experimental semivariogram
# fig.add_trace(go.Scatter(
#     x=lags / 1000,
#     y=gamma_exp,
#     mode="markers",
#     name="Experimental semivariogram",
#     marker=dict(
#         size=9,
#         color="royalblue",
#         symbol="circle",
#         line=dict(width=1, color="black")
#     )
# ))

# # Gaussian fitted model
# fig.add_trace(go.Scatter(
#     x=distances / 1000,
#     y=gamma_theoretical,
#     mode="lines",
#     name="Gaussian model",
#     line=dict(
#         color="firebrick",
#         width=2.5
#     )
# ))

# # ------------------------------------------------------------
# # LAYOUT
# # ------------------------------------------------------------

# fig.update_layout(
#     title=dict(
#         text="Variogram of Mean Seasonal Snow Load",
#         x=0.5,
#         xanchor="center"
#     ),
#     xaxis_title="Lag distance [km]",
#     yaxis_title="Semivariance",
#     template="plotly_white",
#     width=950,
#     height=600,
#     font=dict(
#         family="Arial",
#         size=13
#     ),
#     legend=dict(
#         x=0.02,
#         y=0.98,
#         bgcolor="rgba(255,255,255,0.8)"
#     ),
#     margin=dict(l=80, r=40, t=80, b=70)
# )

# fig.update_xaxes(showgrid=True, gridcolor="lightgrey")
# fig.update_yaxes(showgrid=True, gridcolor="lightgrey")

# fig.show()

In [17]:
# # ------------------------------------------------------------
# # VARIOGRAM OF Sigma SEASONAL SNOW LOAD
# # ------------------------------------------------------------

# valid_sigma = (
#     df_recap["StdDvz"].notna() &
#     np.isfinite(df_recap["StdDvz"])
# )

# df_sigma = df_recap.loc[valid_sigma].copy()

# lats = df_sigma["latitude"].to_numpy(dtype=float)
# lons = df_sigma["longitude"].to_numpy(dtype=float)

# stddevs = df_sigma["StdDvz"].to_numpy(dtype=float)

# R = 6371000

# lat0 = np.mean(lats)
# lon0 = np.mean(lons)

# x_coords = (
#     R * np.cos(np.radians(lat0)) *
#     np.radians(lons - lon0)
# )

# y_coords = R * np.radians(lats - lat0)

# coords_sigma = np.column_stack((x_coords, y_coords))

# # ------------------------------------------------------------
# # VARIOGRAM
# # ------------------------------------------------------------

# nlags = 20

# semivar_model = "gaussian"

# semivar_sigma = Variogram(
#     coords_sigma,
#     stddevs,
#     n_lags=nlags,
#     use_nugget=True,
# )

# semivar_sigma.model = semivar_model

# print(semivar_sigma)

In [18]:
# # ------------------------------------------------------------
# # PLOT SEMIVARIOGRAMMA CON PLOTLY
# # ------------------------------------------------------------


# # Experimental variogram
# lags = nlags
# gamma_exp = semivar_sigma.experimental

# # Theoretical variogram
# max_lag = np.nanmax(lags)
# distances = np.linspace(0, max_lag, 300)
# gamma_theoretical = semivar_sigma.transform(distances)

# # Model parameters
# nugget = semivar_sigma.parameters[2] if semivar_sigma.use_nugget else 0

# # ------------------------------------------------------------
# # FIGURE
# # ------------------------------------------------------------

# fig = go.Figure()

# # Experimental semivariogram
# fig.add_trace(go.Scatter(
#     x=lags / 1000,
#     y=gamma_exp,
#     mode="markers",
#     name="Experimental semivariogram",
#     marker=dict(
#         size=9,
#         color="royalblue",
#         symbol="circle",
#         line=dict(width=1, color="black")
#     )
# ))

# # Gaussian fitted model
# fig.add_trace(go.Scatter(
#     x=distances / 1000,
#     y=gamma_theoretical,
#     mode="lines",
#     name="Gaussian model",
#     line=dict(
#         color="firebrick",
#         width=2.5
#     )
# ))

# # ------------------------------------------------------------
# # LAYOUT
# # ------------------------------------------------------------

# fig.update_layout(
#     title=dict(
#         text="Variogram of Mean Seasonal Snow Load",
#         x=0.5,
#         xanchor="center"
#     ),
#     xaxis_title="Lag distance [km]",
#     yaxis_title="Semivariance",
#     template="plotly_white",
#     width=950,
#     height=600,
#     font=dict(
#         family="Arial",
#         size=13
#     ),
#     legend=dict(
#         x=0.02,
#         y=0.98,
#         bgcolor="rgba(255,255,255,0.8)"
#     ),
#     margin=dict(l=80, r=40, t=80, b=70)
# )

# fig.update_xaxes(showgrid=True, gridcolor="lightgrey")
# fig.update_yaxes(showgrid=True, gridcolor="lightgrey")

# fig.show()

#### Maps

In [19]:
# font = "Arial"

# colorscale = [
#     [0.00, "#ffffff"],   # bianco
#     [0.20, "#deebf7"],
#     [0.40, "#c6dbef"],
#     [0.60, "#9ecae1"],
#     [0.80, "#3182bd"],
#     [1.00, "#08306b"]    # blu scuro
# ]

# # Primo e ultimo anno disponibili
# first_year = seasonal_max["year"].min()
# last_year = seasonal_max["year"].max()

# years = [first_year, last_year]

# fig = make_subplots(
#     rows=1,
#     cols=2,
#     subplot_titles=[f"Winter {y}" for y in years],
#     horizontal_spacing=0.08
# )

# for j, year in enumerate(years):

#     df_year = seasonal_max[seasonal_max["year"] == year]

#     grid = (
#         df_year
#         .pivot(index="latitude",
#                columns="longitude",
#                values="snow_load_max")
#         .sort_index(ascending=True)
#     )

#     fig.add_trace(
#         go.Heatmap(
#             z=grid.values,
#             x=grid.columns,
#             y=grid.index,
#             colorscale=colorscale,
#             zmin=seasonal_max["snow_load_max"].min(),
#             zmax=seasonal_max["snow_load_max"].max(),
#             showscale=(j == 1),
#             colorbar=dict(
#                 title="Snow load<br>(kN/m²)",
#                 tickfont=dict(family=font, size=12),
#                 title_font=dict(family=font, size=14)
#             )
#         ),
#         row=1,
#         col=j + 1
#     )

#     fig.update_xaxes(
#         title="Longitude",
#         showgrid=False,
#         zeroline=False,
#         row=1,
#         col=j + 1
#     )

#     fig.update_yaxes(
#         title="Latitude" if j == 0 else "",
#         showgrid=False,
#         zeroline=False,
#         row=1,
#         col=j + 1
#     )

# fig.update_layout(
#     width=900,
#     height=450,
#     font=dict(family=font, size=14),
#     title=dict(
#         text="Seasonal Maximum Snow Load",
#         x=0.5,
#         xanchor="center",
#         font=dict(family=f"{font}, Bold", size=22)
#     ),
#     margin=dict(l=70, r=70, t=80, b=70)
# )

# fig.show()

##### Massimi annuali

In [20]:
# # ===============================
# # Select the winter to display
# # ===============================
# # year = seasonal_max["year"].min()     # or .max()
# year = 1985

# df_plot = seasonal_max[
#     seasonal_max["year"] == year
# ].copy()

# # Keep only points inside Italy
# df_plot = df_plot.merge(
#     inside_points[["latitude", "longitude"]],
#     on=["latitude", "longitude"],
#     how="inner"
# )

# # ===================================
# # Scatter map coloured by snow load
# # ===================================

# fig = go.Figure()

# fig.add_trace(
#     go.Scattermapbox(
#         lat=df_plot["latitude"],
#         lon=df_plot["longitude"],
#         mode="markers",
#         marker=dict(
#             size=5,
#             color=df_plot["snow_load_max"],
#             colorscale="Blues",
#             cmin=seasonal_max["snow_load_max"].min(),
#             cmax=seasonal_max["snow_load_max"].max(),
#             colorbar=dict(
#                 title="Snow load<br>(kN/m²)"
#             ),
#             opacity=0.95
#         ),
#         text=[
#             f"Winter: {y}<br>"
#             f"Snow load: {v:.2f} kN/m²"
#             for y, v in zip(
#                 df_plot["year"],
#                 df_plot["snow_load_max"]
#             )
#         ],
#         hovertemplate=
#             "Lat: %{lat:.2f}<br>"
#             "Lon: %{lon:.2f}<br>"
#             "%{text}<extra></extra>",
#         showlegend=False
#     )
# )

# # =====================
# # Shapefile boundary
# # =====================

# boundary_geom = shp.geometry.unary_union

# if boundary_geom.geom_type == "Polygon":

#     coords = np.array(boundary_geom.exterior.coords)

#     fig.add_trace(
#         go.Scattermapbox(
#             lat=coords[:,1],
#             lon=coords[:,0],
#             mode="lines",
#             line=dict(color="black", width=2),
#             hoverinfo="skip",
#             showlegend=False
#         )
#     )

# elif boundary_geom.geom_type == "MultiPolygon":

#     for poly in boundary_geom.geoms:

#         coords = np.array(poly.exterior.coords)

#         fig.add_trace(
#             go.Scattermapbox(
#                 lat=coords[:,1],
#                 lon=coords[:,0],
#                 mode="lines",
#                 line=dict(color="black", width=2),
#                 hoverinfo="skip",
#                 showlegend=False
#             )
#         )

# fig.update_layout(
#     mapbox_style="white-bg",
#     mapbox_zoom=5,
#     mapbox_center=dict(lat=42.7, lon=12.5),
#     margin=dict(l=0,r=0,b=0,t=50),
#     title=f"Seasonal maximum snow load ({year})"
# )

# # fig.show()


# # EXPORT HTML
# html_file = rf"D:\DOTTORATO\21_CERRA\1_cerra_massimi_annuali_mappe\cerra_yearly_max_{year}.html"

# fig.write_html(
#     html_file,
#     full_html=True,
#     include_plotlyjs=True,
#     config={"scrollZoom": True,"displayModeBar": True,"responsive": True})

# print(f"HTML exported: {html_file}")

# # EXPORT PDF
# pdf_file = fr"D:\DOTTORATO\21_CERRA\1_cerra_massimi_annuali_mappe\cerra_yearly_max_{year}.pdf"
# fig.write_image(pdf_file,format="pdf", width=1600,height=1100,scale=2)

# print(f"PDF exported: {pdf_file}")

In [21]:
# # ===============================
# # Select the winter to display
# # ===============================
# year = seasonal_max["year"].min()     # or .max()

# df_plot = seasonal_max[
#     seasonal_max["year"] == year
# ].copy()

# # Keep only points inside Italy
# df_plot = df_plot.merge(
#     inside_points[["latitude", "longitude"]],
#     on=["latitude", "longitude"],
#     how="inner"
# )

# # ===================================
# # Scatter map coloured by snow load
# # ===================================

# fig = go.Figure()

# threshold = 3.0  # kN/m²

# df_blue = df_plot[df_plot["snow_load_max"] <= threshold]
# df_black = df_plot[df_plot["snow_load_max"] > threshold]

# # ===================================
# # Blue points: values below threshold
# # ===================================

# fig.add_trace(
#     go.Scattermapbox(
#         lat=df_blue["latitude"],
#         lon=df_blue["longitude"],
#         mode="markers",
#         marker=dict(
#             size=5,
#             color=df_blue["snow_load_max"],
#             colorscale="Blues",
#             cmin=0,
#             cmax=threshold,
#             colorbar=dict(
#                 title="Snow load<br>(kN/m²)"
#             ),
#             opacity=0.95
#         ),
#         text=df_blue["snow_load_max"].round(2),
#         hovertemplate=
#             "Lat: %{lat:.2f}<br>"
#             "Lon: %{lon:.2f}<br>"
#             "Snow load: %{text} kN/m²<extra></extra>",
#         showlegend=False
#     )
# )

# # ===================================
# # Black points: values above threshold
# # ===================================

# fig.add_trace(
#     go.Scattermapbox(
#         lat=df_black["latitude"],
#         lon=df_black["longitude"],
#         mode="markers",
#         marker=dict(
#             size=5,
#             color="black",
#             opacity=0.95
#         ),
#         text=df_black["snow_load_max"].round(2),
#         hovertemplate=
#             "Lat: %{lat:.2f}<br>"
#             "Lon: %{lon:.2f}<br>"
#             "Snow load: %{text} kN/m²<extra></extra>",
#         showlegend=False
#     )
# )

# # =====================
# # Shapefile boundary
# # =====================

# boundary_geom = shp.geometry.unary_union

# if boundary_geom.geom_type == "Polygon":

#     coords = np.array(boundary_geom.exterior.coords)

#     fig.add_trace(
#         go.Scattermapbox(
#             lat=coords[:,1],
#             lon=coords[:,0],
#             mode="lines",
#             line=dict(color="black", width=2),
#             hoverinfo="skip",
#             showlegend=False
#         )
#     )

# elif boundary_geom.geom_type == "MultiPolygon":

#     for poly in boundary_geom.geoms:

#         coords = np.array(poly.exterior.coords)

#         fig.add_trace(
#             go.Scattermapbox(
#                 lat=coords[:,1],
#                 lon=coords[:,0],
#                 mode="lines",
#                 line=dict(color="black", width=2),
#                 hoverinfo="skip",
#                 showlegend=False
#             )
#         )

# fig.update_layout(
#     mapbox_style="white-bg",
#     mapbox_zoom=5,
#     mapbox_center=dict(lat=42.7, lon=12.5),
#     margin=dict(l=0,r=0,b=0,t=50),
#     title=f"Seasonal maximum snow load ({year})"
# )

# fig.show()


##### Valori caratteristici

In [22]:
# ===============================
# Characteristic values (R50) to display
# ===============================
df_R50 = pd.DataFrame({
    "latitude":  data_ev["latitude"].values.ravel(),
    "longitude": data_ev["longitude"].values.ravel(),
    "R50":       np.asarray(R50).ravel()
})

df_plot = df_R50.dropna(subset=["R50"]).copy()

# Keep only points inside Italy
df_plot = df_plot.merge(
    inside_points[["latitude", "longitude"]],
    on=["latitude", "longitude"],
    how="inner"
)

# ===================================
# Scatter map coloured by R50
# ===================================

fig = go.Figure()

fig.add_trace(
    go.Scattermapbox(
        lat=df_plot["latitude"],
        lon=df_plot["longitude"],
        mode="markers",
        marker=dict(
            size=5,
            color=df_plot["R50"],
            colorscale="Blues",
            cmin=df_plot["R50"].min(),
            cmax=df_plot["R50"].max(),
            colorbar=dict(
                title="R50<br>(kN/m²)"
            ),
            opacity=0.95
        ),
        text=[f"R50: {v:.2f} kN/m²" for v in df_plot["R50"]],
        hovertemplate=
            "Lat: %{lat:.2f}<br>"
            "Lon: %{lon:.2f}<br>"
            "%{text}<extra></extra>",
        showlegend=False
    )
)

# =====================
# Shapefile boundary
# =====================

boundary_geom = shp.geometry.unary_union

if boundary_geom.geom_type == "Polygon":

    coords = np.array(boundary_geom.exterior.coords)

    fig.add_trace(
        go.Scattermapbox(
            lat=coords[:,1],
            lon=coords[:,0],
            mode="lines",
            line=dict(color="black", width=2),
            hoverinfo="skip",
            showlegend=False
        )
    )

elif boundary_geom.geom_type == "MultiPolygon":

    for poly in boundary_geom.geoms:

        coords = np.array(poly.exterior.coords)

        fig.add_trace(
            go.Scattermapbox(
                lat=coords[:,1],
                lon=coords[:,0],
                mode="lines",
                line=dict(color="black", width=2),
                hoverinfo="skip",
                showlegend=False
            )
        )

fig.update_layout(
    mapbox_style="white-bg",
    mapbox_zoom=5,
    mapbox_center=dict(lat=42.7, lon=12.5),
    margin=dict(l=0,r=0,b=0,t=50),
    title="Characteristic snow load R50 (T = 50 years)"
)

fig.show()

# # EXPORT HTML
# html_file = r"D:\DOTTORATO\21_CERRA\2_cerra_val_caratt_mappe\cerra_R50.html"

# fig.write_html(
#     html_file,
#     full_html=True,
#     include_plotlyjs=True,
#     config={"scrollZoom": True,"displayModeBar": True,"responsive": True})

# print(f"HTML exported: {html_file}")

# # EXPORT PDF
# pdf_file = r"D:\DOTTORATO\21_CERRA\2_cerra_val_caratt_mappe\cerra_R50.pdf"
# fig.write_image(pdf_file,format="pdf", width=1600,height=1100,scale=2)

# print(f"PDF exported: {pdf_file}")